# Terraform Associate (004) — Prompt 11: Terraform Modules

> **Exam Objective 5**: Understand how to use and create modules, source types, variable scope, and versioning.

## Sections
1. [What Is a Module?](#1-what-is-a-module)
2. [Module Sources](#2-module-sources)
3. [The `module` Block](#3-the-module-block)
4. [Module Versioning](#4-module-versioning)
5. [Variable Scope and Encapsulation](#5-variable-scope-and-encapsulation)
6. [Standard Module Structure](#6-standard-module-structure)
7. [Module Composition](#7-module-composition)
8. [Exam-Style Questions](#8-exam-style-questions)
9. [Real-World Scenarios](#9-real-world-scenarios)

---
## 1. What Is a Module?

A **module** is a container for multiple Terraform resources that are used together. Grouping resources into modules enables reuse, composition, and a cleaner separation of concerns.

### Module Types

| Type | Description |
|---|---|
| **Root module** | The working directory where `terraform init` / `terraform apply` is run. Every Terraform config is at minimum a root module. |
| **Child module** | A module called from another module using a `module` block. Can be local or remote. |
| **Published module** | A module stored in the [Terraform Public Registry](https://registry.terraform.io) or a private registry (HCP Terraform / Terraform Enterprise). |

### Key Facts
- **Every Terraform configuration is a module** — the one in your working directory is the root module.
- A root module can call one or more child modules.
- Child modules can call their own child modules (nested modules).
- Modules promote **DRY** (Don't Repeat Yourself) — define infrastructure once, instantiate many times.

---
## 2. Module Sources

The `source` argument in a `module` block tells Terraform where to find the module code. Terraform supports multiple source types.

### Source Type Reference

| Source Type | Example | Notes |
|---|---|---|
| **Local path** | `source = "./modules/vpc"` | Must start with `./` or `../`. No network access. |
| **Terraform Registry** | `source = "hashicorp/consul/aws"` | Format: `namespace/module/provider`. Supports `version`. |
| **GitHub (HTTPS)** | `source = "github.com/org/repo"` | Fetches default branch. |
| **GitHub (subdir)** | `source = "github.com/org/repo//subdir"` | Double-slash `//` separates repo from subdirectory. |
| **Generic Git (HTTPS)** | `source = "git::https://example.com/repo.git"` | Any HTTPS-accessible Git repo. |
| **Generic Git (SSH)** | `source = "git::ssh://git@example.com/repo.git"` | Uses SSH key authentication. |
| **HTTP archive** | `source = "https://example.com/module.zip"` | `.zip` or `.tar.gz` archive over HTTP/S. |
| **S3 bucket** | `source = "s3::https://s3.amazonaws.com/bucket/module.zip"` | AWS S3. Uses AWS provider credentials. |
| **GCS bucket** | `source = "gcs::https://www.googleapis.com/storage/v1/bucket/module.zip"` | Google Cloud Storage. |

> ⭐ **Exam Tip**: The double-slash `//` in GitHub and Git URLs is the separator between the repository root and a **subdirectory**. It is NOT a typo — it is required syntax.

### Code Example — Source Types

In [ ]:
# Local path source — relative to the calling module's directory
# ---------------------------------------------------------------
module "networking" {
  source = "./modules/networking"
}

# Terraform Registry source — namespace/module/provider
# -------------------------------------------------------
module "vpc" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "~> 5.0"
}

# GitHub source — entire repo
# ----------------------------
module "security_groups" {
  source = "github.com/my-org/terraform-modules"
}

# GitHub source — specific subdirectory (double-slash required)
# -------------------------------------------------------------
module "alb" {
  source = "github.com/my-org/terraform-modules//alb"
}

# GitHub with specific tag/branch ref
# ------------------------------------
module "rds" {
  source = "github.com/my-org/terraform-modules//rds?ref=v2.1.0"
}

---
## 3. The `module` Block

A `module` block calls a child module and passes configuration to it.

### Syntax

```hcl
module "<label>" {
  source  = "<module_source>"
  version = "<version_constraint>"  # registry sources only

  # Input variable values (defined in the child module's variables.tf)
  variable_name = value

  # Meta-arguments
  depends_on = [<resource_or_module>]
  providers  = { aws = aws.us-east-1 }
}
```

### Arguments

| Argument | Required? | Description |
|---|---|---|
| `source` | ✅ Yes | Where to find the module. |
| `version` | For registry only | Version constraint string. Ignored for local/git sources. |
| Input variables | As needed | Any argument not matching a meta-argument is treated as an input variable value. |
| `depends_on` | Optional | Force explicit ordering when implicit dependency cannot be detected. |
| `providers` | Optional | Map of provider aliases to pass into the module. |
| `count` | Optional | Create multiple instances of the module. |
| `for_each` | Optional | Create one instance per element of a map or set. |

> ⭐ **Exam Tip**: `version` is **only supported** for modules sourced from a registry. Specifying `version` with a local path or a direct Git URL has no effect or causes an error.

In [ ]:
# Calling a module from the Terraform Registry with version pinning
module "vpc" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "~> 5.0"            # accepts 5.x but not 6.0+

  # Input variables defined in the VPC module
  name             = "my-vpc"
  cidr             = "10.0.0.0/16"
  azs              = ["us-east-1a", "us-east-1b", "us-east-1c"]
  private_subnets  = ["10.0.1.0/24", "10.0.2.0/24", "10.0.3.0/24"]
  public_subnets   = ["10.0.101.0/24", "10.0.102.0/24", "10.0.103.0/24"]

  enable_nat_gateway = true
  single_nat_gateway = true

  tags = {
    Environment = "production"
    ManagedBy   = "terraform"
  }
}

# Calling a local module
module "app_server" {
  source = "./modules/ec2-instance"

  # Pass input variables
  instance_type = "t3.medium"
  subnet_id     = module.vpc.private_subnets[0]  # output from vpc module
  name          = "web-app"
}

---
## 4. Module Versioning

Versioning is **only available for modules sourced from a registry** (Terraform Public Registry or a private registry). It is not supported for local paths or raw Git URLs.

### Version Constraint Operators

The same operators used for provider version constraints also apply to modules:

| Operator | Meaning | Example |
|---|---|---|
| `= 1.0.0` | Exact version | `version = "= 1.0.0"` |
| `!= 1.0.0` | Exclude version | `version = "!= 1.0.0"` |
| `>= 1.0.0` | At least | `version = ">= 1.0.0"` |
| `<= 1.0.0` | At most | `version = "<= 1.0.0"` |
| `~> 1.0` | Allows only rightmost increment | `version = "~> 1.0"` allows 1.x but not 2.0 |
| `~> 1.0.0` | Patch-level only | `version = "~> 1.0.0"` allows 1.0.x but not 1.1 |

### How Terraform Handles Module Versions

1. `terraform init` downloads and caches module source code in `.terraform/modules/`.
2. The `.terraform.lock.hcl` file records resolved provider versions but **does not** lock module versions from the registry.
3. `terraform init -upgrade` re-evaluates constraints and downloads newer compatible versions.

> ⭐ **Exam Tip**: After adding or changing the `source` or `version` of any module, you **must** run `terraform init` before `terraform plan` or `terraform apply`.

In [ ]:
# Version pinning examples

# Exact version — most strict
module "vpc_exact" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "= 5.1.2"
}

# Pessimistic constraint — allows patch updates within 5.1.x
module "vpc_patch" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "~> 5.1.0"
}

# Pessimistic constraint — allows all 5.x minor releases
module "vpc_minor" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "~> 5.0"
}

# Minimum version floor
module "vpc_floor" {
  source  = "terraform-aws-modules/vpc/aws"
  version = ">= 5.0, < 6.0"
}

# After changing version, run:
#   terraform init
# To pull newer compatible versions, run:
#   terraform init -upgrade

---
## 5. Variable Scope and Encapsulation

Modules have **strict encapsulation** — they do not inherit variables from the calling module. Communication between a caller and a child module uses explicit inputs and outputs.

### How Data Flows

```
Root Module
│
│  module "child" {          ← calls child module
│    source   = "./child"
│    var_name = "value"      ← passes INPUT to child
│  }
│
│  module.child.output_name  ← reads OUTPUT from child
│
└── child/
      variable "var_name" {} ← INPUT declaration
      output  "output_name" {} ← OUTPUT declaration
      resource "aws_..." "..." { ... }   ← internal resource
```

### Rules

| Rule | Detail |
|---|---|
| **Module inputs** | Declared with `variable` blocks *inside* the module. Caller sets values as arguments in the `module` block. |
| **Module outputs** | Declared with `output` blocks *inside* the module. Caller reads them as `module.<name>.<output>`. |
| **No implicit inheritance** | Root module variables are NOT visible inside child modules unless passed explicitly. |
| **Resources not addressable from outside** | You cannot reference `aws_instance.web` inside a module from the root. Use outputs instead. |
| **Targeting** | `module.<name>.<resource_type>.<resource_name>` syntax is used with `-target` only, not for attribute references. |

> ⭐ **Exam Tip**: If a child module needs a value from the root, it must be passed explicitly as an input variable. There is no way to read a root-level `variable` block from inside a child module without passing it through.

In [ ]:
# ----- modules/ec2-instance/variables.tf -----
variable "instance_type" {
  description = "EC2 instance type"
  type        = string
  default     = "t3.micro"
}

variable "subnet_id" {
  description = "Subnet in which to place the instance"
  type        = string
}

variable "name" {
  description = "Name tag for the instance"
  type        = string
}

# ----- modules/ec2-instance/main.tf -----
resource "aws_instance" "this" {
  ami           = data.aws_ami.amazon_linux.id
  instance_type = var.instance_type
  subnet_id     = var.subnet_id

  tags = {
    Name = var.name
  }
}

data "aws_ami" "amazon_linux" {
  most_recent = true
  owners      = ["amazon"]
  filter {
    name   = "name"
    values = ["amzn2-ami-hvm-*-x86_64-gp2"]
  }
}

# ----- modules/ec2-instance/outputs.tf -----
output "instance_id" {
  description = "The ID of the created EC2 instance"
  value       = aws_instance.this.id
}

output "private_ip" {
  description = "Private IP address of the instance"
  value       = aws_instance.this.private_ip
}

In [ ]:
# ----- Root module: main.tf -----
module "web_server" {
  source        = "./modules/ec2-instance"
  instance_type = "t3.medium"               # overrides default
  subnet_id     = aws_subnet.public.id      # passes a root-level resource attribute
  name          = "web-server-prod"
}

# Accessing module outputs from the root module
output "web_server_id" {
  value = module.web_server.instance_id     # module.<name>.<output_name>
}

output "web_server_ip" {
  value = module.web_server.private_ip
}

# WRONG — cannot reference a resource inside a module like this for attribute access:
# resource.aws_instance.web_server.id  ← ERROR
#
# For -target only (not for attribute references):
# terraform apply -target=module.web_server.aws_instance.this

---
## 6. Standard Module Structure

HashiCorp defines a **recommended file structure** for modules. Terraform does not enforce it but the exam tests awareness of it.

```
my-module/
├── main.tf          # Primary resource declarations
├── variables.tf     # Input variable declarations
├── outputs.tf       # Output value declarations
├── README.md        # Module documentation (required for registry publishing)
└── versions.tf      # terraform {} block with required_providers and required_version
```

### File Purposes

| File | Contents |
|---|---|
| `main.tf` | Resource blocks and data sources that form the core functionality. |
| `variables.tf` | All `variable` blocks — input declarations for the module. |
| `outputs.tf` | All `output` blocks — values the module exposes to callers. |
| `README.md` | Human-readable documentation. Required to publish to the Terraform Registry. |
| `versions.tf` | `terraform {}` block containing `required_version` and `required_providers` constraints. |

> ⭐ **Exam Tip**: `README.md` is required to publish a module to the **Terraform Public Registry**. Without it, the registry will not accept the module.

Additional files sometimes used in larger modules:

| File | Purpose |
|---|---|
| `locals.tf` | Local value declarations |
| `data.tf` | Data source blocks |
| `providers.tf` | Provider configuration (usually only in root module) |
| `examples/` | Directory containing usage examples |
| `tests/` | Automated tests using `terraform test` framework |

In [ ]:
# ----- versions.tf -----
terraform {
  required_version = ">= 1.3.0"

  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = ">= 5.0"
    }
  }
}

# ----- variables.tf -----
variable "environment" {
  description = "Deployment environment (dev, staging, prod)"
  type        = string

  validation {
    condition     = contains(["dev", "staging", "prod"], var.environment)
    error_message = "environment must be one of: dev, staging, prod."
  }
}

variable "vpc_cidr" {
  description = "CIDR block for the VPC"
  type        = string
  default     = "10.0.0.0/16"
}

# ----- outputs.tf -----
output "vpc_id" {
  description = "The ID of the created VPC"
  value       = aws_vpc.main.id
}

output "private_subnet_ids" {
  description = "List of private subnet IDs"
  value       = aws_subnet.private[*].id
}

---
## 7. Module Composition

**Module composition** means passing the output of one module as the input to another. This is how complex infrastructure is assembled from reusable building blocks.

### Pattern

```
Root
 ├── module "network"  →  outputs: vpc_id, subnet_ids
 ├── module "compute"  ←  inputs: vpc_id, subnet_ids  (from network)
 └── module "database" ←  inputs: subnet_ids, sg_id   (from network/compute)
```

Terraform automatically creates implicit dependencies when module outputs are used as inputs — the `network` module will complete before `compute` starts because `compute` references `module.network.vpc_id`.

In [ ]:
# Root module: main.tf — Module Composition Example

# Layer 1: Network (VPC, subnets, routing)
module "network" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "~> 5.0"

  name            = "app-vpc"
  cidr            = "10.0.0.0/16"
  azs             = ["us-east-1a", "us-east-1b"]
  private_subnets = ["10.0.1.0/24", "10.0.2.0/24"]
  public_subnets  = ["10.0.101.0/24", "10.0.102.0/24"]
}

# Layer 2: Compute — uses outputs from network module
module "compute" {
  source = "./modules/compute"

  # Pass network outputs as compute inputs
  vpc_id     = module.network.vpc_id             # module output → module input
  subnet_ids = module.network.private_subnets    # module output → module input
  environment = var.environment
}

# Layer 3: Database — uses outputs from both modules
module "database" {
  source = "./modules/rds"

  subnet_ids = module.network.private_subnets    # from network
  sg_id      = module.compute.app_sg_id          # from compute
  db_name    = "appdb"
}

# Root outputs — expose information from all layers
output "load_balancer_dns" {
  value = module.compute.alb_dns_name
}

output "database_endpoint" {
  value = module.database.endpoint
}

---
## 8. Exam-Style Questions

Test your understanding before checking answers.

### Question 1

A developer wants to call a module stored in a GitHub repository under the subdirectory `modules/vpc`. Which `source` argument correctly references this subdirectory?

- A. `source = "github.com/org/repo/modules/vpc"`
- B. `source = "github.com/org/repo//modules/vpc"`
- C. `source = "github.com/org/repo:modules/vpc"`
- D. `source = "git::github.com/org/repo/modules/vpc"`

<details><summary>Answer</summary>

**B. `source = "github.com/org/repo//modules/vpc"`**

The double-slash `//` is the required separator between the repository root and a subdirectory path in any Git-based module source (GitHub, GitLab, generic Git). A single slash `/` is interpreted as part of the repository URL path, not as a subdirectory separator. Options C and D use incorrect syntax for GitHub sources.

</details>

### Question 2

A root module has a variable `region` with a value of `"us-east-1"`. A child module is called from the root module. How can the child module access the value `"us-east-1"`?

- A. Child modules automatically inherit all root module variables.
- B. The child module can read `var.region` directly because it shares the same variable namespace.
- C. The root module must pass the value explicitly as an input argument in the `module` block.
- D. The child module must re-declare the variable with the same name and Terraform will sync the value.

<details><summary>Answer</summary>

**C. The root module must pass the value explicitly as an input argument in the `module` block.**

Terraform modules are **fully encapsulated**. A child module has no access to its caller's variables, locals, or resources unless the caller explicitly passes them as input arguments. The child module must declare a corresponding `variable` block and the root must pass the value:

```hcl
module "child" {
  source = "./child"
  region = var.region   # explicit pass-through
}
```

</details>

### Question 3

A module block specifies `source = "./modules/networking"` and `version = "~> 2.0"`. What happens when Terraform processes this configuration?

- A. Terraform uses version `~> 2.0` to download the correct version from the local directory.
- B. Terraform throws an error because `version` is not supported for local path sources.
- C. Terraform ignores the `version` argument for local sources and uses the files in the path.
- D. Terraform checks `versions.tf` inside `./modules/networking` and validates compatibility.

<details><summary>Answer</summary>

**B. Terraform throws an error because `version` is not supported for local path sources.**

The `version` argument is only supported for modules installed from a **registry** (Terraform Public Registry or a private registry). Local path sources (`./...` or `../...`) do not have a version concept. Similarly, direct Git/GitHub sources do not support the `version` argument — you control the version for Git sources with the `ref` query parameter (e.g., `?ref=v2.0.0`). Using `version` with a local path results in an error.

</details>

### Question 4

A module named `"network"` creates an `aws_vpc` resource with the local name `"main"`. The root module needs the VPC ID to pass to another module. Which of the following correctly accesses the VPC ID?

- A. `aws_vpc.main.id`
- B. `module.network.aws_vpc.main.id`
- C. `module.network.vpc_id` (assuming the module exports an output named `vpc_id`)
- D. `var.network.vpc_id`

<details><summary>Answer</summary>

**C. `module.network.vpc_id` (assuming the module exports an output named `vpc_id`)**

Resources inside a child module are **not directly addressable** from outside the module. To expose a resource attribute, the module must declare an `output` block, and the caller reads it via `module.<module_name>.<output_name>`. Option A would only work if `aws_vpc.main` existed in the root module scope. Option B is the syntax used with `terraform apply -target` for targeting, not for reading attribute values.

Inside `network/outputs.tf`:
```hcl
output "vpc_id" {
  value = aws_vpc.main.id
}
```
In root module:
```hcl
module.network.vpc_id  # correct
```

</details>

---
## 9. Real-World Scenarios

Five practical scenarios demonstrating module concepts tested on the exam.

<details>
<summary>Scenario 1: Adopting a Community VPC Module from the Terraform Registry</summary>

**Situation**: Your team is building an AWS environment and wants to use the popular `terraform-aws-modules/vpc/aws` community module instead of writing custom VPC code. You need to pin the version to avoid unexpected breaking changes.

**Exam Sub-Objective**: Module sources — Terraform Registry format; module versioning; `terraform init`.

**HCL Configuration**:
```hcl
# main.tf
module "vpc" {
  source  = "terraform-aws-modules/vpc/aws"   # namespace/module/provider
  version = "~> 5.1"                          # allows 5.1.x but not 5.2 or 6.0

  name = "production-vpc"
  cidr = "10.100.0.0/16"

  azs             = ["eu-west-1a", "eu-west-1b", "eu-west-1c"]
  private_subnets = ["10.100.1.0/24", "10.100.2.0/24", "10.100.3.0/24"]
  public_subnets  = ["10.100.101.0/24", "10.100.102.0/24", "10.100.103.0/24"]

  enable_nat_gateway   = true
  single_nat_gateway   = false  # one NAT GW per AZ for HA
  enable_dns_hostnames = true
}

output "vpc_id" {
  value = module.vpc.vpc_id
}
```

**CLI Commands**:
```bash
terraform init              # downloads module into .terraform/modules/
terraform plan              # previews VPC resources the module will create
terraform apply             # creates all VPC resources

# Later, to update to latest 5.x patch:
terraform init -upgrade
```

**Expected Outcome**:
- `terraform init` fetches the pinned version from `registry.terraform.io` and caches it locally.
- A full VPC stack (subnets, route tables, internet gateway, NAT gateways) is created.
- `module.vpc.vpc_id` returns the created VPC ID.

**Key Takeaway**: Registry source format is `namespace/module/provider`. The `version` constraint uses the same operators as provider versions. `terraform init` must be run whenever `source` or `version` changes.

</details>

<details>
<summary>Scenario 2: Creating and Calling a Local Reusable Module</summary>

**Situation**: Your team repeatedly creates similarly configured S3 buckets across multiple projects. You want to create an internal local module to standardize bucket creation (versioning, encryption, public-access-block) and call it multiple times from the root module.

**Exam Sub-Objective**: Local path source; standard module structure; module inputs and outputs; encapsulation.

**Module Directory Structure**:
```
project/
├── main.tf
├── variables.tf
├── outputs.tf
└── modules/
    └── s3-bucket/
        ├── main.tf
        ├── variables.tf
        ├── outputs.tf
        └── versions.tf
```

**HCL — Module (modules/s3-bucket/variables.tf)**:
```hcl
variable "bucket_name" {
  type = string
}

variable "environment" {
  type    = string
  default = "dev"
}
```

**HCL — Module (modules/s3-bucket/main.tf)**:
```hcl
resource "aws_s3_bucket" "this" {
  bucket = var.bucket_name
  tags   = { Environment = var.environment }
}

resource "aws_s3_bucket_versioning" "this" {
  bucket = aws_s3_bucket.this.id
  versioning_configuration { status = "Enabled" }
}

resource "aws_s3_bucket_public_access_block" "this" {
  bucket = aws_s3_bucket.this.id
  block_public_acls       = true
  block_public_policy     = true
  ignore_public_acls      = true
  restrict_public_buckets = true
}
```

**HCL — Module (modules/s3-bucket/outputs.tf)**:
```hcl
output "bucket_id" {
  value = aws_s3_bucket.this.id
}

output "bucket_arn" {
  value = aws_s3_bucket.this.arn
}
```

**HCL — Root Module (main.tf)**:
```hcl
module "logs_bucket" {
  source      = "./modules/s3-bucket"     # local path — ./ required
  bucket_name = "my-app-logs-${var.environment}"
  environment = var.environment
}

module "assets_bucket" {
  source      = "./modules/s3-bucket"     # same module, different instance
  bucket_name = "my-app-assets-${var.environment}"
  environment = var.environment
}

output "logs_bucket_arn" {
  value = module.logs_bucket.bucket_arn
}
```

**Expected Outcome**: Two independently configured S3 buckets with consistent security settings. The module is instantiated twice from the same source path.

**Key Takeaway**: Local path sources require `./` prefix. The same module can be called multiple times with different labels and input values.

</details>

<details>
<summary>Scenario 3: Composing Multiple Modules (Network → Compute → Database)</summary>

**Situation**: You need to deploy a three-tier application on AWS: VPC networking, an EC2 application tier, and an RDS database. Each tier is managed by a separate module, and each tier depends on outputs from the previous tier.

**Exam Sub-Objective**: Module composition; implicit dependencies via module outputs; accessing module outputs.

**HCL Configuration (root main.tf)**:
```hcl
module "network" {
  source  = "terraform-aws-modules/vpc/aws"
  version = "~> 5.0"
  name    = "app-vpc"
  cidr    = "10.0.0.0/16"
  azs             = ["us-east-1a", "us-east-1b"]
  private_subnets = ["10.0.1.0/24", "10.0.2.0/24"]
  public_subnets  = ["10.0.101.0/24", "10.0.102.0/24"]
}

module "compute" {
  source = "./modules/compute"
  # Outputs from network become inputs to compute
  # Terraform detects implicit dependency: compute waits for network
  vpc_id          = module.network.vpc_id
  public_subnets  = module.network.public_subnets
  private_subnets = module.network.private_subnets
  instance_type   = "t3.medium"
}

module "database" {
  source = "./modules/rds"
  # Depends on both network and compute outputs
  subnet_ids      = module.network.private_subnets
  app_sg_id       = module.compute.app_security_group_id
  db_name         = "appdb"
  engine_version  = "15.3"
}

output "app_url" {
  value = "https://${module.compute.alb_dns_name}"
}
```

**Expected Outcome**:
- Terraform builds a DAG: `network` → `compute` → `database`.
- Each module completes before the dependent module starts.
- The root module exposes just the final URL — all implementation details are encapsulated in modules.

**Key Takeaway**: Module outputs used as inputs to another module create implicit ordering dependencies. Terraform handles this automatically without needing `depends_on`.

</details>

<details>
<summary>Scenario 4: Pinning a Module Version and Upgrading Safely</summary>

**Situation**: A team uses `~> 3.0` for a community module. A new minor version `3.5.0` is released with a bug fix they need. They want to update to at least `3.5.0` while staying in the 3.x line, then later lock to an exact version for production.

**Exam Sub-Objective**: Module versioning constraints; `terraform init -upgrade`; safe version pinning strategies.

**HCL Progression**:
```hcl
# Stage 1: Open constraint during development
module "ecs" {
  source  = "terraform-aws-modules/ecs/aws"
  version = "~> 3.0"    # allows 3.x releases
  cluster_name = "app-cluster"
}

# Stage 2: Floor constraint to get the bug fix
module "ecs" {
  source  = "terraform-aws-modules/ecs/aws"
  version = ">= 3.5.0, < 4.0.0"   # requires 3.5.0+ but not 4.x
  cluster_name = "app-cluster"
}

# Stage 3: Exact pin for production stability
module "ecs" {
  source  = "terraform-aws-modules/ecs/aws"
  version = "= 3.5.2"   # exact version — no surprises
  cluster_name = "app-cluster"
}
```

**CLI Commands**:
```bash
# After changing version constraint, re-initialize to fetch new version
terraform init -upgrade

# Check what version was downloaded
cat .terraform/modules/modules.json

terraform plan  # verify no unintended changes
```

**Expected Outcome**:
- `~> 3.0` allows any 3.x version; running `terraform init -upgrade` pulls the latest 3.x available.
- `>= 3.5.0, < 4.0.0` ensures the bug fix is included while preventing 4.x breaking changes.
- `= 3.5.2` locks production to a specific, tested version.

**Key Takeaway**: Version constraints for modules work identically to provider version constraints. `terraform init -upgrade` is needed to fetch newer compatible module versions.

</details>

<details>
<summary>Scenario 5: Multi-Region Deployment Using Module with `providers` Meta-Argument</summary>

**Situation**: You need to deploy identical infrastructure in two AWS regions (`us-east-1` and `eu-west-1`) using the same local module. The `providers` meta-argument passes aliased provider configurations into the module.

**Exam Sub-Objective**: `providers` meta-argument for modules; provider aliases; module encapsulation across provider configurations.

**HCL Configuration**:
```hcl
# providers.tf — root module defines aliased providers
provider "aws" {
  region = "us-east-1"
  alias  = "us_east"
}

provider "aws" {
  region = "eu-west-1"
  alias  = "eu_west"
}

# main.tf — call same module twice with different providers
module "us_east_infra" {
  source = "./modules/app-infra"

  region      = "us-east-1"
  environment = var.environment

  providers = {
    aws = aws.us_east    # pass aliased provider to module
  }
}

module "eu_west_infra" {
  source = "./modules/app-infra"

  region      = "eu-west-1"
  environment = var.environment

  providers = {
    aws = aws.eu_west    # same module, different provider
  }
}
```

**Inside the module (modules/app-infra/versions.tf)**:
```hcl
terraform {
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = ">= 5.0"
    }
  }
}

# Note: the module does NOT define a provider block.
# It receives the provider from the caller via the providers meta-argument.
# This is the recommended pattern for reusable modules.
```

**Expected Outcome**:
- Identical infrastructure is created in two regions from the same module code.
- Resources inside `module.us_east_infra` are deployed to `us-east-1`.
- Resources inside `module.eu_west_infra` are deployed to `eu-west-1`.
- The module itself has no provider configuration — it is injected from the root.

**Key Takeaway**: The `providers` meta-argument maps provider aliases from the root into the module's required providers. Reusable modules should declare `required_providers` but not configure providers — configuration is the root's responsibility.

</details>

---
## Quick-Reference Cheat Sheet

### Module Source Quick Reference

| Source Type | `source` Syntax | `version` Supported? |
|---|---|---|
| Local path | `"./modules/vpc"` | ❌ No |
| Parent directory | `"../shared/vpc"` | ❌ No |
| Terraform Registry | `"namespace/module/provider"` | ✅ Yes |
| GitHub (root) | `"github.com/org/repo"` | ❌ No (use `?ref=`) |
| GitHub (subdir) | `"github.com/org/repo//subdir"` | ❌ No (use `?ref=`) |
| Generic Git HTTPS | `"git::https://..."` | ❌ No (use `?ref=`) |
| Generic Git SSH | `"git::ssh://..."` | ❌ No (use `?ref=`) |

### Module Scope Rules

| Scenario | How to Do It |
|---|---|
| Pass value to child module | Declare `variable` in child; pass value as arg in `module` block |
| Read value from child module | Declare `output` in child; read as `module.<name>.<output>` in caller |
| Child reads root variable | Not possible directly — must pass explicitly as input |
| Reference resource in child module | Use outputs; direct resource reference across module boundary is not allowed |
| Target resource in child with CLI | `terraform apply -target=module.<name>.<type>.<resource_name>` |

### Commands Summary

```bash
terraform init             # Download/update modules and providers
terraform init -upgrade    # Re-evaluate version constraints, download newer compatible versions
terraform get              # Download modules only (without reinitializing providers)
terraform get -update      # Force re-download of modules
```

### Standard Module Files

```
main.tf        — resources
variables.tf   — inputs
outputs.tf     — exposed values
versions.tf    — required_providers + required_version
README.md      — documentation (required for registry publishing)
```

### Most-Tested Module Facts

1. **`version` only works with registry sources** — not local paths, not raw Git URLs.
2. **Double slash `//`** separates repo from subdirectory in Git-based sources.
3. **`terraform init` required** after any change to `source` or `version`.
4. **No implicit variable inheritance** — child modules are fully encapsulated.
5. **`module.<name>.<output>`** is the only way to read child module data from the root.
6. **`README.md` required** to publish a module to the Terraform Public Registry.
7. **`providers` meta-argument** passes aliased provider configurations into a module.